# 10 · Preload: Left statistics before the Right side is ready

**When:** the Left data is ready long before the Right side. You compute the Stage 2 payloads
(one per pair × window × column) now, keep them in a preload area, and a later run picks up the
ones that are still valid.

**What is preloaded and what is not**

| Stage | Preloaded? | Why |
| --- | --- | --- |
| Stage 1 (partition counts, reconciliation) | **no**, it always runs live | it is cheap, and its fresh counts are what proves a preloaded unit is still valid |
| Stage 2 (column statistics) | **yes** | this is the expensive pull + statistics part |

**Layout** (a folder or an S3 prefix outside every run, e.g. `s3://<bucket>/dtrack/preload/`):

```text
preload/<pair>/units/<window>/<COLUMN>.json    stats + partition_counts + fingerprint + produced_at
preload/<pair>/_history/<time>/<window>/<COLUMN>.json   every overwritten version
```

Rules decided for version 1:
- the pair keeps its own `vintage`;
- fixed-N sampling cannot be preloaded (the keys depend on the final windows);
- the source is assumed immutable; a rare value-only fix is re-uploaded explicitly for one unit.

In [ ]:
import json

import handoff
import nbkit
import pandas as pd
from nbkit import show, table, tree

## 1 · A Left-only config is enough

With nothing known about the Right side, the full `validate_config` refuses the pair (no Right
side, no `col_map`). The preload does not need it: it takes a `columns` list, fills a placeholder
Right side internally, and the fingerprint ignores that placeholder. The real run later uses the
full config; units are matched by **left** column name and get their Right names from that run.

In [ ]:
from dtrack_left.cli.config import validate_config

folder = nbkit.sandbox("10")
data, config, _ = nbkit.dataset(folder)
orders = config["pairs"]["orders"]
left_only = {
    "pairs": {
        "orders": {
            "left": orders["left"],
            "columns": list(orders["col_map"]),
            "key": orders["key"],
            "key_families": orders["key_families"],
            "sample": orders["sample"],
            "vintage": orders["vintage"],
            "fromDate": orders["fromDate"],
            "toDate": orders["toDate"],
        }
    }
}
try:
    validate_config(json.loads(json.dumps(left_only)))
except ValueError as exc:
    print("validate_config:", exc)
pair = handoff.pair_from_config(left_only, "orders")
preload = nbkit.VolumeStore(folder / "preload")

## 2 · Day 1: only January is ready

`until` is the cutoff: nothing after it is preloaded. `preload_status` lists every unit the
preload should hold; `preload_units` computes the missing and stale ones.

In [ ]:
january = handoff.left_counts(pair, until="2026-01-31")
table(handoff.preload_status(preload, "orders", pair, january))
table(handoff.preload_units(preload, "orders", pair, folder / "work", january))
tree(preload.root)

In [ ]:
unit = preload.get_json("orders/units/2026-01-01/AMOUNT.json")
show(
    {k: v for k, v in unit.items() if k != "partition_counts"}
    | {"partition_counts": f"{len(unit['partition_counts'])} days"},
    "one preloaded unit",
)

## 3 · Day 2: February arrives, top up only what is missing

In [ ]:
everything = handoff.left_counts(pair)
table(handoff.preload_status(preload, "orders", pair, everything))
table(handoff.preload_units(preload, "orders", pair, folder / "work", everything))

## 4 · The source changed: counts give it away

Rows were added to 2026-01-05 after the preload. The next status check sees different partition
counts for the January window and marks it stale; the top-up recomputes it and keeps the old
version in `_history/`.

In [ ]:
left_csv = data / "left" / "ORDERS.csv"
frame = pd.read_csv(left_csv, dtype=str)
extra = frame[frame.BUSINESS_DATE == "2026-01-05"].head(2).assign(ORDER_ID=["999001", "999002"])
pd.concat([frame, extra]).to_csv(left_csv, index=False)

changed = handoff.left_counts(pair)
table(handoff.preload_status(preload, "orders", pair, changed))
table(handoff.preload_units(preload, "orders", pair, folder / "work", changed))
tree(preload.root, "orders/_history")

## 5 · A value-only fix: re-upload one unit on purpose

If a value is corrected but no row is added or removed, the counts cannot tell. The person who
fixed the data re-uploads exactly the affected unit with `force={window: [columns]}`.

In [ ]:
table(
    handoff.preload_units(
        preload, "orders", pair, folder / "work", changed, force={"2026-02-01": ["AMOUNT"]}
    )
)

## 6 · Known matched partitions

If you already know which partitions will match (from an earlier run, or a quick count on the
Right system), preload only those with `partitions=[...]`. The windows then have the same holes
the real manifest will have, and every unit is adoptable (notebook 11).

## 7 · Fixed-N sampling is refused

In [ ]:
customers = handoff.pair_from_config(
    {"pairs": {"customers": {**config["pairs"]["customers"]}}}, "customers"
)
try:
    handoff.preload_status(preload, "customers", customers, handoff.left_counts(customers))
except ValueError as exc:
    print(exc)